In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchvision.transforms.functional as tvF
from sklearn import svm
from mpl_toolkits.axes_grid1 import make_axes_locatable

import math

In [2]:
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')


In [3]:
def confusion_cal(y_test, py_test, y_type):
    class_num = len(np.unique(y_test))
    confusion_num = np.zeros((class_num, class_num))
    y_test = torch.tensor(y_test)
    py_test = torch.tensor(py_test)
    if y_type == 0:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(torch.argmax(py_test, dim=1) == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))
    elif y_type == 1:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(py_test == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))

    return confusion_num

In [4]:
dat_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/data'
SPK = []
ptype = []
for rnum in range(0, 30):
    SPK.append(np.load(dat_dir + f'/FixedRepresentation/Spk_an_wn_{rnum}.npy'))
    ptype.append(np.load(dat_dir + f'/FixedRepresentation/Ptype_an_wn_{rnum}.npy'))

SPK = np.concatenate(SPK, axis=0)
ptype = np.concatenate(ptype, axis=0)

SPK1 = []
ptype1 = []
for rnum in range(0, 49):
    SPK1.append(np.load(dat_dir + f'/FixedRepresentation01/Spk_an_wn_{rnum}.npy'))
    ptype1.append(np.load(dat_dir + f'/FixedRepresentation01/Ptype_an_wn_{rnum}.npy'))

SPK1 = np.concatenate(SPK1, axis=0)
ptype1 = np.concatenate(ptype1, axis=0)

In [5]:
acu_set = np.zeros((50,6))
acu_set_test = np.zeros((50,6))
for ii in range(0, 50):

    train_ind = np.random.choice(np.size(SPK,axis=0), np.int32(0.8 * np.size(SPK,axis=0)), replace=False)
    test_ind = np.setdiff1d(np.arange(0, np.size(SPK,axis=0)), train_ind)
    test_ind_od = np.random.choice(np.size(SPK1,axis=0), np.int32(0.2 * np.size(SPK1,axis=0)), replace=False)

    x_train = SPK[train_ind,:]
    y_train = ptype[train_ind]

    x_test = SPK[test_ind,:]
    y_test = ptype[test_ind]

    x_test_od = SPK1[test_ind_od,:]
    y_test_od = ptype1[test_ind_od]

    # Initialize model
    model = svm.LinearSVC()

    # Fit the model to training data
    model.fit(x_train, y_train)

    # Check test set accuracy
    acu_set[ii,0] = model.score(x_test, y_test)
    acu_set_test[ii,0] = model.score(x_test_od, y_test_od)

    for jj in range(0,5):
        acu_set[ii,jj + 1] = model.score(x_test[np.argwhere(y_test == jj).flatten()], y_test[np.argwhere(y_test == jj).flatten()])
        acu_set_test[ii,jj + 1] = model.score(x_test_od[np.argwhere(y_test_od == jj).flatten()], y_test_od[np.argwhere(y_test_od == jj).flatten()])


In [6]:
save_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/result'
np.save(save_dir + '/svm_fixedrepresentation.npy', acu_set)
np.save(save_dir + '/svm_fixedrepresentation1.npy', acu_set_test)
np.save(save_dir + '/svm_fixedrepresentation_confusion_num.npy', confusion_cal(y_test, model.predict(x_test), y_type=1))
np.save(save_dir + '/svm_fixedrepresentation_confusion_num1.npy', confusion_cal(y_test_od, model.predict(x_test_od), y_type=1))

In [7]:
acu_set

array([[0.59090909, 0.71317829, 0.384     , 0.392     , 0.47321429,
        0.976     ],
       [0.61525974, 0.7079646 , 0.46363636, 0.40944882, 0.52816901,
        0.97580645],
       [0.57792208, 0.72033898, 0.41353383, 0.38931298, 0.44642857,
        0.94262295],
       [0.57142857, 0.71304348, 0.41044776, 0.328125  , 0.53787879,
        0.95327103],
       [0.64285714, 0.79824561, 0.47413793, 0.3968254 , 0.57142857,
        0.95522388],
       [0.58766234, 0.80733945, 0.375     , 0.3362069 , 0.47014925,
        0.96124031],
       [0.61525974, 0.79069767, 0.46774194, 0.28346457, 0.55855856,
        0.968     ],
       [0.61201299, 0.70588235, 0.41525424, 0.432     , 0.56153846,
        0.94354839],
       [0.60714286, 0.76984127, 0.40944882, 0.38211382, 0.54471545,
        0.94871795],
       [0.60551948, 0.7768595 , 0.45669291, 0.3902439 , 0.49635036,
        0.97222222],
       [0.57142857, 0.76612903, 0.37062937, 0.39090909, 0.37815126,
        0.96666667],
       [0.59090909, 0

In [8]:
np.round(acu_set.mean(axis=0), 3)

array([0.593, 0.736, 0.421, 0.365, 0.487, 0.963])

In [9]:
np.round(acu_set_test.mean(axis=0), 3)

array([0.589, 0.741, 0.421, 0.335, 0.489, 0.951])